# LifeMate — Qwen3.5-4B SFT continuation with English system prompt v1

This notebook continues from the final LoRA adapter produced by the existing two-epoch QLoRA run. It performs one additional epoch with the same training settings as the Vietnamese-prompt continuation, but uses the English `Agent.next_action()` system prompt in the SFT rows. The optimizer and scheduler are fresh. Output is written to a separate folder.

## Attach these Kaggle inputs

1. English-prompt train-only SFT data: `daoviet/lifemate-qwen35-sft-data-en-v1`.
2. Base Transformers model: `limamateus/qwen3-5-4b` (same model version/path as the parent run).
3. Epoch-2 adapter: attach the existing Kaggle Model `lifemate-qwen35-sft-epoch2-adapter-v1` or the adapter package dataset.
4. Offline dependency wheels: keep the existing Kaggle Notebook input `daoviet/lifemate-wheels` attached; Cell 1 installs them without Internet.

The setup selects the SFT file by the expected English-variant SHA-256. The Vietnamese dataset may remain attached; the notebook ignores its different hash. Keep the epoch-2 adapter and base model attached. The BTC competition input may remain attached but is not read by this training notebook.

This is a prompt-language-only data ablation: user transcripts remain Vietnamese; action targets, data rows, train/validation split, and val309 inference inputs are unchanged. The data contains no CoT targets or added CoT checklist. The Qwen chat template is rendered in thinking mode, matching the parent continuation. Use the same 4 × NVIDIA L4 setup, effective batch 16, and 8,192-token cap as the prior run. Internet may remain disabled.


In [ ]:
import subprocess
import sys
import glob

# Kaggle Notebook input daoviet/lifemate-wheels; install locally with Internet disabled.
wheels = glob.glob("/kaggle/input/notebooks/**/*.whl", recursive=True)
print("Found offline wheels:")
for wheel in wheels:
    print("-", wheel)
if not wheels:
    raise FileNotFoundError("No offline .whl files found. Attach daoviet/lifemate-wheels as a Notebook input.")
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "-q", *wheels], check=True)
print("Offline wheel installation complete.")


In [ ]:
from importlib.metadata import version

EXPECTED = {"transformers": "5.22.1", "bitsandbytes": "0.50.2", "accelerate": "1.15.0", "peft": "0.21.2"}
for package, expected in EXPECTED.items():
    actual = version(package)
    print(package, actual)
    if actual != expected:
        raise RuntimeError(f"Expected {package}=={expected}, found {actual}")


In [ ]:
import hashlib, json, os, random, subprocess, sys, time
from pathlib import Path
from importlib.metadata import version
import torch
from transformers import AutoTokenizer

SEED = 20261106
NUM_EPOCHS = 1
LEARNING_RATE = 1e-5
MAX_SEQ_LENGTH = 8192
EFFECTIVE_BATCH_TARGET = 16
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/qwen35_lifemate_sft_v1_prompt_en_continuation_epoch3')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if any(OUTPUT_ROOT.iterdir()):
    raise RuntimeError(f'Output folder already contains a prior run: {OUTPUT_ROOT}. Change OUTPUT_ROOT to a new run folder; do not overwrite an earlier adapter.')
EXPECTED_ADAPTER_SHA256 = 'a94a027cd48f8c4c5e7c45108418cdbc9265bba5cebd4595c04a18bdd4ec2611'
EXPECTED_SFT_SHA256 = 'f2255f7055c8dc4ea09faecd06c7d97eb745434640a9ae8c6812c9b2e9b97195'
SFT_PROMPT_VARIANT = 'english-system-prompt-v1'

def unique_path(paths, label):
    paths = sorted(set(Path(p).resolve() for p in paths))
    if len(paths) != 1:
        raise RuntimeError(f'Expected exactly one {label}, found {len(paths)}: {paths[:8]}')
    return paths[0]

sft_candidates = sorted(set(INPUT_ROOT.rglob('sft_train.jsonl')))
sft_matches = [p for p in sft_candidates if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_SFT_SHA256]
if len(sft_matches) != 1:
    raise RuntimeError(f'Expected exactly one English-prompt SFT file with SHA-256 {EXPECTED_SFT_SHA256}; found {len(sft_matches)}. Attach the English dataset daoviet/lifemate-qwen35-sft-data-en-v1.')
sft_file = sft_matches[0].resolve()
SFT_DIR = sft_file.parent
SFT_SHA256 = EXPECTED_SFT_SHA256
SFT_MANIFEST_FILE = SFT_DIR / 'dataset_manifest.json'
if not SFT_MANIFEST_FILE.is_file():
    raise FileNotFoundError(f'Missing English SFT manifest: {SFT_MANIFEST_FILE}')
SFT_MANIFEST = json.loads(SFT_MANIFEST_FILE.read_text(encoding='utf-8'))
if SFT_MANIFEST.get('dataset_variant') != SFT_PROMPT_VARIANT or SFT_MANIFEST.get('prompt_language') != 'en' or SFT_MANIFEST.get('cot_targets_added') is not False:
    raise RuntimeError(f'Unexpected SFT data variant in manifest: {SFT_MANIFEST_FILE}')
adapter_weight_file = unique_path(INPUT_ROOT.rglob('adapter_model.safetensors'), 'epoch-2 adapter_model.safetensors')
ADAPTER_DIR = adapter_weight_file.parent
ADAPTER_CONFIG_FILE = ADAPTER_DIR / 'adapter_config.json'
if not ADAPTER_CONFIG_FILE.is_file():
    raise FileNotFoundError(f'Missing adapter_config.json next to {adapter_weight_file}')
ADAPTER_SHA256 = hashlib.sha256(adapter_weight_file.read_bytes()).hexdigest()
if ADAPTER_SHA256 != EXPECTED_ADAPTER_SHA256:
    raise RuntimeError(f'Parent adapter hash mismatch: expected {EXPECTED_ADAPTER_SHA256}, got {ADAPTER_SHA256}')
ADAPTER_CONFIG = json.loads(ADAPTER_CONFIG_FILE.read_text(encoding='utf-8'))
if ADAPTER_CONFIG.get('base_model_name_or_path', '').replace('\\', '/') != '/kaggle/input/models/limamateus/qwen3-5-4b/pytorch/default/1/Qwen3.5-4B':
    raise RuntimeError(f'Adapter was trained from an unexpected base model: {ADAPTER_CONFIG.get("base_model_name_or_path")}')
if ADAPTER_CONFIG.get('r') != 16 or ADAPTER_CONFIG.get('lora_alpha') != 32 or ADAPTER_CONFIG.get('peft_type') != 'LORA':
    raise RuntimeError('Unexpected parent adapter configuration; refusing to continue training.')
VAL_INPUTS_FILE = SFT_DIR / 'val309_inputs.jsonl'
TRAIN_IDS_FILE = SFT_DIR / 'train_ids.json'
VAL_IDS_FILE = SFT_DIR / 'val309_ids.json'
if not all(p.is_file() for p in [VAL_INPUTS_FILE, TRAIN_IDS_FILE, VAL_IDS_FILE]):
    raise FileNotFoundError(f'SFT dataset bundle is incomplete: {SFT_DIR}')

# This SFT continuation uses only the prepared SFT bundle; BTC train files are not needed here.

model_configs = []
for p in INPUT_ROOT.rglob('config.json'):
    path_text = str(p.parent).lower().replace('_', '-')
    try:
        cfg = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        continue
    arch = ' '.join(cfg.get('architectures', [])).lower().replace('_', '-')
    model_type = str(cfg.get('model_type', '')).lower().replace('_', '-')
    if 'qwen3-5' in path_text and '4b' in path_text and ('qwen3-5' in arch or 'qwen3-5' in model_type):
        model_configs.append(p)
MODEL_DIR = unique_path(model_configs, 'Qwen3.5-4B model config').parent

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 1:
    raise RuntimeError('Kaggle GPU is not enabled. Select a GPU accelerator before running training.')
try:
    TRANSFORMERS_VERSION = version('transformers')
    PEFT_VERSION = version('peft')
    BNB_VERSION = version('bitsandbytes')
except Exception as exc:
    raise RuntimeError('This runtime needs Transformers 5.x, PEFT, Accelerate, and bitsandbytes available offline.') from exc

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print({'SFT_DIR': str(SFT_DIR), 'SFT_PROMPT_VARIANT': SFT_PROMPT_VARIANT, 'MODEL_DIR': str(MODEL_DIR), 'ADAPTER_DIR': str(ADAPTER_DIR), 'adapter_sha256': ADAPTER_SHA256, 'sft_sha256': SFT_SHA256, 'gpus': [{'name': torch.cuda.get_device_name(i), 'vram_gib': round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1), 'bf16': torch.cuda.is_bf16_supported()} for i in range(GPU_COUNT)], 'transformers': TRANSFORMERS_VERSION, 'peft': PEFT_VERSION, 'bitsandbytes': BNB_VERSION})


In [ ]:
# Audit and tokenize; only SFT train rows are passed to Trainer.
with sft_file.open(encoding='utf-8') as f:
    SFT_ROWS = [json.loads(line) for line in f if line.strip()]
TRAIN_IDS = set(json.loads(TRAIN_IDS_FILE.read_text(encoding='utf-8')))
VAL_IDS = set(json.loads(VAL_IDS_FILE.read_text(encoding='utf-8')))
assert len(TRAIN_IDS) == 1236 and len(VAL_IDS) == 309 and not TRAIN_IDS & VAL_IDS
assert len(SFT_ROWS) == 3873, f'Expected 3873 train-only action rows, found {len(SFT_ROWS)}'
assert {r['sample_id'] for r in SFT_ROWS} <= TRAIN_IDS
assert all(len(r['messages']) == 2 and r['messages'][0]['role'] == 'system' and r['messages'][1]['role'] == 'user' for r in SFT_ROWS)
assert all(r['target'].get('action') in {'tool_call', 'stop'} for r in SFT_ROWS)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

THINK_CLOSE = '</think>\n\n'
encoded_rows = []
truncated_count = 0
lengths = []
for row in SFT_ROWS:
    prompt = tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=True, enable_thinking=True)
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    close_ids = tokenizer.encode(THINK_CLOSE, add_special_tokens=False)
    answer_text = json.dumps(row['target'], ensure_ascii=False, separators=(',', ':'))
    answer_ids = tokenizer.encode(answer_text, add_special_tokens=False) + [tokenizer.eos_token_id]
    room_for_prompt = MAX_SEQ_LENGTH - len(close_ids) - len(answer_ids)
    if room_for_prompt < 1:
        raise ValueError(f'Target exceeds MAX_SEQ_LENGTH: {row["sample_id"]}/{row["step_index"]}')
    if len(prompt_ids) > room_for_prompt:
        truncated_count += 1
        keep_head = int(room_for_prompt * 0.72)
        prompt_ids = prompt_ids[:keep_head] + prompt_ids[-(room_for_prompt - keep_head):]
    input_ids = prompt_ids + close_ids + answer_ids
    labels = [-100] * (len(prompt_ids) + len(close_ids)) + answer_ids
    encoded_rows.append({'input_ids': input_ids, 'labels': labels})
    lengths.append(len(input_ids))

TOKENIZED_PATH = Path('/tmp/lifemate_qwen35_sft_tokens.pt')
torch.save(encoded_rows, TOKENIZED_PATH)
print({'sft_sha256': SFT_SHA256, 'train_trajectories': len(TRAIN_IDS), 'val309_excluded_from_training': len(VAL_IDS), 'sft_rows': len(SFT_ROWS), 'target_action_counts': {}, 'sequence_p50': sorted(lengths)[len(lengths)//2], 'sequence_p95': sorted(lengths)[int(0.95*(len(lengths)-1))], 'sequence_max': max(lengths), 'truncated_prompts': truncated_count, 'truncated_pct': round(100*truncated_count/len(SFT_ROWS), 2), 'token_file_mb': round(TOKENIZED_PATH.stat().st_size/2**20, 1)})


In [ ]:
# One process per visible GPU. Each worker loads the base model and resumes adapter weights only.
WORKER_SCRIPT = r'''
import argparse, json, os, random, sys
from pathlib import Path
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForMultimodalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import PeftModel, prepare_model_for_kbit_training

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', required=True); ap.add_argument('--adapter', required=True); ap.add_argument('--tokens', required=True); ap.add_argument('--out', required=True)
    ap.add_argument('--epochs', type=float, required=True); ap.add_argument('--lr', type=float, required=True)
    ap.add_argument('--max-length', type=int, required=True); ap.add_argument('--grad-accum', type=int, required=True)
    ap.add_argument('--warmup-steps', type=int, required=True); ap.add_argument('--save-steps', type=int, required=True)
    ap.add_argument('--seed', type=int, required=True); ap.add_argument('--adapter-sha256', required=True); ap.add_argument('--sft-sha256', required=True); ap.add_argument('--sft-prompt-variant', required=True); args = ap.parse_args()
    os.environ.setdefault('HF_HUB_OFFLINE','1'); os.environ.setdefault('TRANSFORMERS_OFFLINE','1'); os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
    rank=int(os.environ.get('LOCAL_RANK','0')); world=int(os.environ.get('WORLD_SIZE','1')); torch.cuda.set_device(rank)
    random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tokenizer=AutoTokenizer.from_pretrained(args.model,local_files_only=True,use_fast=True)
    if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
    rows=torch.load(args.tokens,map_location='cpu',weights_only=True)
    class Encoded(Dataset):
        def __len__(self): return len(rows)
        def __getitem__(self,i): return rows[i]
    class Collator:
        def __call__(self,features):
            m=max(len(x['input_ids']) for x in features); m=((m+7)//8)*8
            ids=[]; masks=[]; labels=[]
            for x in features:
                n=m-len(x['input_ids']); ids.append(x['input_ids']+[tokenizer.pad_token_id]*n); masks.append([1]*len(x['input_ids'])+[0]*n); labels.append(x['labels']+[-100]*n)
            labels_t=torch.tensor(labels,dtype=torch.long)
            if labels_t.shape[0] != 1: raise RuntimeError('Selective logits collator currently requires per-device batch size 1')
            shifted=labels_t[:,1:]; keep_mask=shifted.ne(-100)
            logits_to_keep=torch.where(keep_mask[0])[0]
            shift_labels=shifted[keep_mask]
            if logits_to_keep.numel() == 0: raise RuntimeError('Batch contains no supervised next-token targets')
            return {'input_ids':torch.tensor(ids,dtype=torch.long),'attention_mask':torch.tensor(masks,dtype=torch.long),'labels':labels_t,'logits_to_keep':logits_to_keep,'shift_labels':shift_labels}
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
    model_kwargs={'quantization_config':bnb,'device_map':{'':rank},'local_files_only':True,'low_cpu_mem_usage':True,'attn_implementation':'sdpa'}
    import transformers, peft, bitsandbytes
    model_kwargs['dtype' if int(transformers.__version__.split('.')[0])>=5 else 'torch_dtype']=dtype
    model=AutoModelForMultimodalLM.from_pretrained(args.model,**model_kwargs)
    model.config.use_cache=False
    model=prepare_model_for_kbit_training(model,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False})
    model=PeftModel.from_pretrained(model,args.adapter,is_trainable=True,local_files_only=True)
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    if trainable <= 0: raise RuntimeError('Loaded parent adapter has no trainable parameters; check is_trainable=True and adapter files')
    if rank==0: model.print_trainable_parameters()
    model.train(); probe_i=max(range(len(rows)),key=lambda i:len(rows[i]['input_ids'])); probe=Collator()([rows[probe_i]])
    if rank==0: print(f"Preflighting longest training sequence: {len(rows[probe_i]['input_ids'])} tokens on {torch.cuda.get_device_name(rank)}",flush=True)
    probe={k:v.to(rank) for k,v in probe.items()}; loss=model(**probe).loss
    if not torch.isfinite(loss): raise RuntimeError('QLoRA continuation preflight returned non-finite loss')
    loss.backward(); model.zero_grad(set_to_none=True); del probe,loss; torch.cuda.empty_cache()
    if rank==0: print('QLoRA continuation preflight passed on all visible ranks',flush=True)
    out=Path(args.out); checkpoints=out/'checkpoints'; checkpoints.mkdir(parents=True,exist_ok=True)
    training_kwargs=dict(output_dir=str(checkpoints),num_train_epochs=args.epochs,per_device_train_batch_size=1,gradient_accumulation_steps=args.grad_accum,learning_rate=args.lr,weight_decay=0.01,warmup_steps=args.warmup_steps,lr_scheduler_type='cosine',logging_steps=10,save_strategy='steps',save_steps=args.save_steps,save_total_limit=5,optim='adamw_torch',bf16=dtype==torch.bfloat16,fp16=dtype==torch.float16,gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},ddp_find_unused_parameters=False if world>1 else None,remove_unused_columns=False,report_to=[],seed=args.seed,data_seed=args.seed)
    trainer=Trainer(model=model,args=TrainingArguments(**training_kwargs),train_dataset=Encoded(),data_collator=Collator())
    result=trainer.train()
    adapter=out/'adapter_final'; trainer.save_model(str(adapter))
    if trainer.is_world_process_zero():
        tokenizer.save_pretrained(adapter)
        report={'run_type':'continuation_from_epoch2_adapter_english_prompt','continuation_source_adapter_sha256':args.adapter_sha256,'sft_train_sha256':args.sft_sha256,'sft_prompt_variant':args.sft_prompt_variant,'epochs_requested':args.epochs,'learning_rate':args.lr,'optimizer_state':'fresh','scheduler_state':'fresh cosine schedule','warmup_steps':args.warmup_steps,'checkpoint_interval_steps':args.save_steps,'seed':args.seed,'world_size':world,'effective_batch_size':world*args.grad_accum,'max_seq_length':args.max_length,'train_rows':len(rows),'trainable_parameters':trainable,'torch_version':torch.__version__,'transformers_version':transformers.__version__,'peft_version':peft.__version__,'bitsandbytes_version':bitsandbytes.__version__,'train_runtime_seconds':result.metrics.get('train_runtime'),'train_loss':result.metrics.get('train_loss'),'log_history':trainer.state.log_history}
        (out/'training_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
        print(json.dumps({k:v for k,v in report.items() if k!='log_history'},indent=2),flush=True)
        if not list(adapter.glob('adapter_model*.safetensors')): raise RuntimeError('Continuation adapter weights were not saved')
        print('ADAPTER_READY',str(adapter),flush=True)
if __name__=='__main__': main()
'''
WORKER_PATH = OUTPUT_ROOT / 'train_qwen35_lora_continuation.py'
WORKER_PATH.write_text(WORKER_SCRIPT, encoding='utf-8')
grad_accum = max(1, (EFFECTIVE_BATCH_TARGET + GPU_COUNT - 1) // GPU_COUNT)
steps_per_epoch = (len(SFT_ROWS) + GPU_COUNT * grad_accum - 1) // (GPU_COUNT * grad_accum)
warmup_steps = max(1, round(0.05 * steps_per_epoch))
print(f"GPU_COUNT={GPU_COUNT}; grad_accum={grad_accum}; effective_batch={GPU_COUNT * grad_accum}; planned_steps={steps_per_epoch}; warmup_steps={warmup_steps}")
command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node', str(GPU_COUNT), str(WORKER_PATH), '--model', str(MODEL_DIR), '--adapter', str(ADAPTER_DIR), '--tokens', str(TOKENIZED_PATH), '--out', str(OUTPUT_ROOT), '--epochs', str(NUM_EPOCHS), '--lr', str(LEARNING_RATE), '--max-length', str(MAX_SEQ_LENGTH), '--grad-accum', str(grad_accum), '--warmup-steps', str(warmup_steps), '--save-steps', '60', '--seed', str(SEED), '--adapter-sha256', ADAPTER_SHA256, '--sft-sha256', SFT_SHA256, '--sft-prompt-variant', SFT_PROMPT_VARIANT]
print('Launching QLoRA continuation:', ' '.join(command))
subprocess.run(command, check=True)
TOKENIZED_PATH.unlink(missing_ok=True)
print('Continuation training complete. New adapter:', OUTPUT_ROOT / 'adapter_final')


## After training

The new output is `/kaggle/working/qwen35_lifemate_sft_v1_prompt_en_continuation_epoch3/`. It contains `adapter_final/`, checkpoints saved every 60 optimizer steps, and `training_report.json`. Download it as a new artifact; the parent epoch-2 adapter dataset remains unchanged.

The report records the parent adapter hash, English SFT-data hash and variant name, training settings, package versions, runtime, and loss. No validation labels are used during training. After download, evaluate the new adapter using the fixed downstream cohort and full val309 setup with the established inference configuration before deciding whether it improves the task metric.

If the run fails before `ADAPTER_READY`, inspect the last traceback. A failed run leaves its output directory in place to prevent accidental overwrite; change `OUTPUT_ROOT` to a fresh path before retrying. This notebook does not merge adapters, convert GGUF, create Kaggle datasets, publish a kernel, or submit to the competition.
